# Week 4 Assignment - Convolutional VAE on MNIST

Submission notebook structured exactly according to the assignment requirements.

## Part 1 - Train the Model

### 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


### 2. Load MNIST

In [ ]:
batch_size = 128
latent_dim = 20
learning_rate = 1e-3
epochs = 30
beta = 1.0

transform = transforms.ToTensor()

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


### 3. ConvVAE Architecture

In [ ]:
class ConvVAE(nn.Module):

    def __init__(self, latent_dim=20):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1,32,3,2,1),
            nn.ReLU(),
            nn.Conv2d(32,64,3,2,1),
            nn.ReLU(),
            nn.Conv2d(64,128,3,2,0),
            nn.ReLU(),
            nn.Flatten()
        )

        self.fc_mu = nn.Linear(128*3*3, latent_dim)
        self.fc_logvar = nn.Linear(128*3*3, latent_dim)

        self.decoder_input = nn.Linear(latent_dim, 128*3*3)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128,64,4,2,1),
            nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,1,4,2,1),
            nn.Sigmoid()
        )

    def encode(self,x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self,mu,logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self,z):
        h = self.decoder_input(z)
        h = h.view(-1,128,3,3)
        return self.decoder(h)

    def forward(self,x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu,logvar)
        recon = self.decode(z)
        return recon, mu, logvar


### 4. Loss Function

In [ ]:
def vae_loss(recon,x,mu,logvar,beta=1.0):

    recon = F.interpolate(recon,size=(28,28),mode='bilinear')

    bce = F.binary_cross_entropy(recon,x,reduction='sum')

    kld = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    total = bce + beta * kld

    return total,bce,kld


### 5. Training Functions

In [ ]:
def train_epoch(model,loader,optimizer,beta):

    model.train()

    total_loss = 0
    total_bce = 0
    total_kld = 0

    for x,_ in loader:

        x = x.to(device)

        optimizer.zero_grad()

        recon,mu,logvar = model(x)

        loss,bce,kld = vae_loss(recon,x,mu,logvar,beta)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_bce += bce.item()
        total_kld += kld.item()

    n = len(loader.dataset)

    return total_loss/n,total_bce/n,total_kld/n


@torch.no_grad()
def evaluate(model,loader,beta):

    model.eval()

    total_loss = 0
    total_bce = 0
    total_kld = 0

    for x,_ in loader:

        x = x.to(device)

        recon,mu,logvar = model(x)

        loss,bce,kld = vae_loss(recon,x,mu,logvar,beta)

        total_loss += loss.item()
        total_bce += bce.item()
        total_kld += kld.item()

    n = len(loader.dataset)

    return total_loss/n,total_bce/n,total_kld/n


### 6. Train for 30 Epochs

In [ ]:
model = ConvVAE(latent_dim).to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

history = {
    'train_loss':[],
    'train_bce':[],
    'train_kld':[],
    'test_loss':[],
    'test_bce':[],
    'test_kld':[]
}

for epoch in range(epochs):

    train_loss,train_bce,train_kld = train_epoch(
        model,train_loader,optimizer,beta
    )

    test_loss,test_bce,test_kld = evaluate(
        model,test_loader,beta
    )

    history['train_loss'].append(train_loss)
    history['train_bce'].append(train_bce)
    history['train_kld'].append(train_kld)
    history['test_loss'].append(test_loss)
    history['test_bce'].append(test_bce)
    history['test_kld'].append(test_kld)

    print(
        f'Epoch {epoch+1}/{epochs} | '
        f'Train Loss: {train_loss:.2f} | '
        f'Test Loss: {test_loss:.2f}'
    )


### 7. BCE and KL Loss Curves

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history['train_loss'],label='Train')
plt.plot(history['test_loss'],label='Test')
plt.title('Total Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history['train_bce'],label='BCE')
plt.plot(history['train_kld'],label='KL')
plt.title('BCE and KL Components')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


# Part 2 - Visualizations

### Reconstructions

In [ ]:
@torch.no_grad()
def show_reconstructions():

    x,_ = next(iter(test_loader))
    x = x[:10].to(device)

    recon,_,_ = model(x)
    recon = F.interpolate(recon,size=(28,28),mode='bilinear')

    fig,ax = plt.subplots(2,10,figsize=(15,3))

    for i in range(10):
        ax[0,i].imshow(x[i,0].cpu(),cmap='gray')
        ax[0,i].axis('off')

        ax[1,i].imshow(recon[i,0].cpu(),cmap='gray')
        ax[1,i].axis('off')

    plt.show()

show_reconstructions()


### Generated Samples

In [ ]:
@torch.no_grad()
def generate_digits(model,n=16):

    z = torch.randn(n,latent_dim).to(device)

    imgs = model.decode(z)
    imgs = F.interpolate(imgs,size=(28,28),mode='bilinear')

    fig,ax = plt.subplots(4,4,figsize=(6,6))

    for i,a in enumerate(ax.flat):
        a.imshow(imgs[i,0].cpu(),cmap='gray')
        a.axis('off')

    plt.show()

generate_digits(model)


### t-SNE of Latent Codes

In [ ]:
@torch.no_grad()
def latent_tsne():

    zs = []
    labels = []

    for x,y in test_loader:

        x = x.to(device)

        mu,_ = model.encode(x)

        zs.append(mu.cpu())
        labels.append(y)

        if len(torch.cat(zs)) >= 2000:
            break

    z = torch.cat(zs)[:2000].numpy()
    y = torch.cat(labels)[:2000].numpy()

    embedding = TSNE(
        n_components=2,
        random_state=42
    ).fit_transform(z)

    plt.figure(figsize=(8,6))
    plt.scatter(
        embedding[:,0],
        embedding[:,1],
        c=y,
        s=5,
        cmap='tab10'
    )
    plt.colorbar()
    plt.show()

latent_tsne()


### Latent Space Interpolation

In [ ]:
@torch.no_grad()
def interpolation():

    x,_ = next(iter(test_loader))

    a = x[0:1].to(device)
    b = x[1:2].to(device)

    mu1,_ = model.encode(a)
    mu2,_ = model.encode(b)

    fig,ax = plt.subplots(1,8,figsize=(12,2))

    for i,t in enumerate(torch.linspace(0,1,8)):

        z = (1-t)*mu1 + t*mu2

        img = model.decode(z)
        img = F.interpolate(img,size=(28,28),mode='bilinear')

        ax[i].imshow(img[0,0].cpu(),cmap='gray')
        ax[i].axis('off')

    plt.show()

interpolation()


# Part 3 - β Experiment

In [ ]:
betas = [0.0,0.5,2.0]

for beta_value in betas:

    print(f'Beta = {beta_value}')

    beta_model = ConvVAE(latent_dim).to(device)

    beta_optimizer = optim.Adam(
        beta_model.parameters(),
        lr=learning_rate
    )

    for epoch in range(10):
        train_epoch(
            beta_model,
            train_loader,
            beta_optimizer,
            beta_value
        )

    generate_digits(beta_model)


### Analysis

As β increases, the KL divergence term receives greater importance and forces the latent distribution to remain closer to a standard normal distribution. This produces a smoother and more structured latent space but usually makes reconstructions and generated samples blurrier. Very large β values can lead to posterior collapse, where the decoder begins ignoring the latent representation and produces similar outputs regardless of the input.
